# Large ViTs: SigLIP 2, DINOv3 and an ensemble

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rivas-vega/A3_Recv/blob/main/finished/colabs/03_vit_semantic.ipynb)

I moved from the distilled CNN to large frozen encoders. I wanted to compare a supervised CIFAR head with the image-text representation already learned during pretraining, then check whether combining the predictions helped.

| Branch | Input and adaptation | Public Kaggle score |
|---|---|---:|
| SigLIP 2 Giant probe | 384×384, supervised linear head | 0.92019 |
| DINOv3 ViT-7B probe | 256×256, head on CLS + mean patch features | 0.92978 |
| SigLIP 2 Giant zero-shot | 384×384, public text templates, full image + flip | 0.94015 |
| Ensemble | 50% zero-shot SigLIP + 25% SigLIP probe + 25% DINO probe | 0.94844 |

Both encoders remain frozen. The text branch uses the 80 public OpenCLIP templates unchanged and the ten CIFAR-10 class names.

This notebook targets an A100 with BF16. Models run sequentially and features are cached in FP16 blocks. DINOv3 requires Hugging Face access; SigLIP can run independently first.


## 1. Dependencies

Use an A100 runtime. The install cell keeps Colab's paired PyTorch and torchvision packages. Restart the runtime if a different Transformers version was already imported.


In [ ]:
%pip -q install "transformers==4.57.3" "huggingface_hub>=0.34,<1" "accelerate>=1.2,<2" kagglehub sentencepiece protobuf

## 2. Load the experiment module

The module revision and SHA256 are fixed so a resumed run uses the same implementation.


In [ ]:
from pathlib import Path
import hashlib, importlib.util, json, os, sys, time
import requests

CODE_REVISION = 'f4402e673211225162c720b3e3276d2c218e73d5'
EXPECTED_SHA256 = '8d3baa53881282e34cbdb21c3527c7164213f3e78c1d921f393193c011958afc'
url = f"https://raw.githubusercontent.com/rivas-vega/A3_Recv/{CODE_REVISION}/semantic_vit.py"
response = requests.get(url, timeout=60)
response.raise_for_status()
payload = response.content
if hashlib.sha256(payload).hexdigest() != EXPECTED_SHA256:
    raise RuntimeError("Module hash mismatch for this notebook revision.")
module_path = Path('/content/semantic_vit.py')
module_path.write_bytes(payload)
spec = importlib.util.spec_from_file_location('semantic_vit', module_path)
sv = importlib.util.module_from_spec(spec)
sys.modules['semantic_vit'] = sv
spec.loader.exec_module(sv)
print('Verified code revision:', CODE_REVISION)

## 3. Run configuration

Keep `RUN_NAME` when resuming, and use a fresh name for a different configuration. I used DINO `7b` for the reported submission; `hplus` and `large` remain available as smaller alternatives.

Images stay at 32×32 in RAM and are resized per batch. Drive stores feature caches and trained heads. The large checkpoint downloads need substantial temporary disk space in Colab.


In [ ]:
from google.colab import drive
import torch

if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime before continuing.')
if not torch.cuda.is_bf16_supported():
    raise RuntimeError('This experiment requires a BF16-capable GPU, such as A100.')
print(torch.cuda.get_device_name(), '| VRAM:', round(torch.cuda.get_device_properties(0).total_memory/2**30,1), 'GiB')
torch.set_float32_matmul_precision('high')
torch.manual_seed(42)

RUN_NAME = 'vit_giant_7b_v1'
SESSION_BUDGET_HOURS = 7.0
SETTINGS = dict(seed=42, dino_variant='7b', siglip_flip=True, siglip_augmix=True)
if SETTINGS['dino_variant'] not in sv.DINO_MODELS:
    raise ValueError('dino_variant must be 7b, hplus or large')
drive.mount('/content/drive')
BASE = Path('/content/drive/MyDrive/A3_Recv')
RUN_DIR = BASE/'runs'/RUN_NAME
CACHE_DIR = BASE/'cache_semantic_vit'
SCRATCH = Path('/content/recvis_vit_chunks')
budget = sv.Budget(SESSION_BUDGET_HOURS)
print('Run directory:', RUN_DIR)

## 4. Kaggle data

Authentication uses `KAGGLE_API_TOKEN` from Colab Secrets, or KaggleHub login if the secret is unavailable. The account must have joined the competition.


In [ ]:
import kagglehub
from google.colab import userdata
if not os.environ.get('KAGGLE_API_TOKEN'):
    try:
        token = userdata.get('KAGGLE_API_TOKEN')
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
        token = None
    if token:
        os.environ['KAGGLE_API_TOKEN'] = token
    del token
kagglehub.login()
DATA_ROOT = Path(kagglehub.competition_download('mva-recvis-2026'))
data = sv.read_data(DATA_ROOT)
experiment = sv.Experiment(data, SETTINGS, RUN_DIR, CACHE_DIR, SCRATCH, budget)
sv.atomic_json(dict(code_revision=CODE_REVISION, module_sha256=EXPECTED_SHA256), RUN_DIR/'code_revision.json')
for split in ('train', 'val', 'test'):
    print(split, data[split].shape)
print('Validation: selection / calibration / evaluation:', [len(x) for x in data['parts']])

## 5. SigLIP: zero-shot and linear probe

First, compare GPU preprocessing against the official processor. If the maximum pixel difference exceeds 0.025 or embedding cosine similarity falls below 0.999, use the reference CPU processor. Features from the two preprocessing paths are cached separately.

The zero-shot branch averages scores from the full image and its horizontal flip. The supervised branch trains a head on frozen features from the training images and an AugMix view: AdamW, at most 80 epochs, patience 10 from epoch 10.

Validation is split by class: 20% for head selection, 20% for temperature calibration and 60% for evaluation. Head selection averages clean and AugMix accuracy. This split is internal to the experiment; the competition validation set was already used in earlier runs.


In [ ]:
experiment.run_siglip()
print('SigLIP complete. Its CSV files can be downloaded before running DINO.')

## 6. Hugging Face access for DINOv3

Use a read token from the account with access to the [ViT-7B checkpoint](https://huggingface.co/facebook/dinov3-vit7b16-pretrain-lvd1689m). `HF_TOKEN` stays empty in the shared copy.


In [ ]:
HF_TOKEN = ""  # DINOv3 read token; leave empty in the shared copy.

## 7. DINOv3 and the ensemble

The DINO head takes the concatenation of normalized CLS and normalized mean patch features, excluding the four register tokens. DINO itself remains frozen.

The fixed ensemble averages calibrated probabilities with weights **0.50 / 0.25 / 0.25** for zero-shot SigLIP, the SigLIP head and the DINO head. These weights are part of the run configuration.

Extraction can pause between blocks and resume with the same `RUN_NAME`. The SigLIP CSV files remain available if the DINO stage is interrupted.


In [ ]:
experiment.run_dino(HF_TOKEN)
print('Experiment complete:', RUN_DIR/'submission_vit_ensemble.csv')

## 8. Results and downloads

The table below reads the saved validation metrics. CSV files can be downloaded separately for Kaggle. `submission_siglip_blend.csv` is an intermediate combination of the two SigLIP branches; the scored three-branch ensemble is `submission_vit_ensemble.csv`.

Zero-shot SigLIP outperformed its supervised head by **1.996 percentage points** on Kaggle. The ensemble added **0.829 points** over zero-shot SigLIP. This made image-text models the most promising next direction.


In [ ]:
import pandas as pd
from IPython.display import display
rows = [json.loads(p.read_text()) for p in sorted(RUN_DIR.glob('*_metrics.json'))]
if rows:
    table = pd.DataFrame(rows)
    columns = [c for c in ('name','accuracy','augmented_audit_accuracy','nll','temperature','kaggle_public_accuracy') if c in table]
    display(table[columns])
else:
    print('No stage has completed yet.')
for p in sorted(RUN_DIR.glob('submission_*.csv')):
    print(p.name, f'({p.stat().st_size/1024:.0f} KB)')

In [ ]:
from google.colab import files
archive = experiment.bundle()
files.download(str(archive))
# Bundle predictions and metrics for this run.

## References and notes

- [SigLIP 2 Giant 384](https://huggingface.co/google/siglip2-giant-opt-patch16-384), [input configuration](https://huggingface.co/google/siglip2-giant-opt-patch16-384/blob/main/preprocessor_config.json).
- [DINOv3 ViT-7B](https://huggingface.co/facebook/dinov3-vit7b16-pretrain-lvd1689m), [image transform](https://github.com/facebookresearch/dinov3#image-transform).
- [OpenCLIP templates at the pinned revision](https://github.com/mlfoundations/open_clip/blob/4cc208d3a44a2fb6d6435006a46f520d2ccc29b1/src/open_clip/zero_shot_metadata.py); license in `OPENCLIP_LICENSE.txt`.

The 256×256 DINO input is my feature-extraction setting, not a claim about a single pretraining resolution. All inputs are resized directly from 32×32. These runs use no histogram matching or test-set adaptation.
